# Core

Data structure for text diff operations

In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from dataclasses import dataclass, field, asdict
from copy import deepcopy
from typing import List, Dict, Any, Optional
import uuid

In [ ]:
#| export 
@dataclass
class Segment:
    id: str
    text: str
    segment_type: str # "keep", "add", "remove"
    explanation: str = ""
    accepted: bool = False

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)
    
    @staticmethod
    def from_dict(data: Dict[str, Any]) -> 'Segment':
        return Segment(**data)


In [ ]:
test_segment_data = {"id": "001",
                "text": "hello",
                "segment_type": "keep",
                "explanation": "",
                "accepted": False}

test_segment = Segment.from_dict(test_segment_data)

assert test_segment.id == test_segment_data["id"]
assert test_segment.text == test_segment_data["text"]
assert test_segment.segment_type == test_segment_data["segment_type"]

assert test_segment.to_dict() == test_segment_data

In [ ]:
#| export
@dataclass
class DocumentState:
    """Complete document state with history tracking."""
    segments: List[Segment] = field(default_factory=list)
    history: List[List[Segment]] = field(default_factory=list)
    history_index: int = -1

    def add_segments(self, segments_data: List[Dict[str, Any]]) -> None:
        """Initialize or replace segments from a list of dicts."""
        self.segments = [Segment.from_dict(data) for data in segments_data]
        self._push_history()

    def edit_segment(self, segment_id: str, new_text: str) -> bool:
        """Edit a segment's text. Returns True if found."""
        segment_index = self._find_segment_index(segment_id)
        if segment_index is None:
            return False
        self.segments[segment_index].text = new_text
        self._push_history()
        return True

    def accept_segment(self, segment_id: str, action: str) -> bool:
        """
        Accept or reject a segment change.

        Actions:
        - 'keep-remove': keep text that was marked for removal
        - 'accept-remove': actually remove the segment
        - 'keep-add': accept an addition (convert to keep)
        - 'reject-add': reject an addition (delete segment)
        - 'remove': for a replace pair, keep the old text
        - 'add': for a replace pair, keep the new text
        """
        segment_index = self._find_segment_index(segment_id)
        if segment_index is None:
            return False

        if action in ("reject-add", "accept-remove"):
            self.segments.pop(segment_index)

        elif action in ("keep-remove", "keep-add"):
            self.segments[segment_index].segment_type = "keep"

        elif action in ("remove", "add"):
            if segment_index + 1 >= len(self.segments):
                return False
            if not self._is_change_pair(self.segments[segment_index], self.segments[segment_index + 1]):
                return False

            if action == "remove":
                kept_text = self.segments[segment_index].text
            else:
                kept_text = self.segments[segment_index + 1].text

            new_segment = Segment(
                id=str(uuid.uuid4()),
                text=kept_text,
                segment_type='keep',
                explanation='',
                accepted=True
            )
            self.segments[segment_index:segment_index + 2] = [new_segment]
        else:
            return False

        self._push_history()
        return True

    def undo(self) -> bool:
        """Undo last change. Returns True if there was something to undo."""
        if len(self.history) == 1 or self.history_index == 0:
            return False
        else:
            self.history_index -= 1
        self.segments = deepcopy(self.history[self.history_index])
        return True

    def redo(self) -> bool:
        """Redo last undone change. Returns True if there was something to redo."""
        if len(self.history) == 1 or self.history_index == len(self.history) - 1:
            return False
        self.history_index += 1
        self.segments = deepcopy(self.history[self.history_index])
        return True

    def get_working_text(self) -> str:
        """Get the current working text (includes kept and removed text)."""
        working_text = ""
        for segment in self.segments:
            if segment.segment_type in ("keep", "remove"):
                working_text += segment.text
        return working_text

    def get_final_text(self) -> str:
        """Get the final text (only accepted/kept segments)."""
        working_text = ""
        for segment in self.segments:
            if segment.segment_type == "keep":
                working_text += segment.text
        return working_text

    def to_dict(self) -> Dict[str, Any]:
        """Serialize the full state including undo/redo availability."""
        return {
            "segments": [seg.to_dict() for seg in self.segments],
            "can_undo": self.history_index > 0,
            "can_redo": self.history_index < len(self.history) - 1,
            "working_text": self.get_working_text(),
            "final_text": self.get_final_text(),
        }

    def _push_history(self) -> None:
        """Snapshot current segments into history, truncating any future states."""
        self.history = self.history[:self.history_index+1] + [deepcopy(self.segments)]
        self.history_index = len(self.history)-1

    def _find_segment_index(self, segment_id: str) -> Optional[int]:
        """Find a segment's index by id. Returns None if not found."""
        for i, seg in enumerate(self.segments):
            if seg.id == segment_id:
                return i
        return None

    def _is_change_pair(self, seg1: Segment, seg2: Segment) -> bool:
        """Check if two segments form a remove+add (or add+remove) pair."""
        return (seg1.segment_type == "remove" and seg2.segment_type == "add")

In [ ]:

# Test data: simulates a diff where "The old sentence." becomes "The new sentence."
test_segments_data = [
    {"id": "s1", "text": "The ", "segment_type": "keep", "explanation": "", "accepted": True},
    {"id": "s2", "text": "old ", "segment_type": "remove", "explanation": "replacing word", "accepted": False},
    {"id": "s3", "text": "new ", "segment_type": "add", "explanation": "replacing word", "accepted": False},
    {"id": "s4", "text": "sentence.", "segment_type": "keep", "explanation": "", "accepted": True},
]

doc = DocumentState()
doc.add_segments(test_segments_data)

assert doc.get_working_text() == "The old sentence."
assert doc.get_final_text() == "The sentence."

# test accept replace pair (keep the new text)
doc.accept_segment("s2", "add")
assert doc.get_working_text() == "The new sentence."

# test edit segment
doc.edit_segment("s1", "A ")
assert doc.get_working_text() == "A new sentence."

# test undo
doc.undo()
assert doc.get_working_text() == "The new sentence."

# test redo
doc.redo()
assert doc.get_working_text() == "A new sentence."

# test to_dict output
output = doc.to_dict()
assert output["working_text"] == "A new sentence."
assert output["final_text"] == "A new sentence."
assert output["can_undo"] == True
assert output["can_redo"] == False
assert len(output["segments"]) == 3
assert output["segments"][0]["text"] == "A "
assert output["segments"][0]["segment_type"] == "keep"

In [ ]:
# Test standalone remove: "I really like cats" where "really " is proposed for removal
remove_data = [
    {"id": "r1", "text": "I ", "segment_type": "keep", "explanation": "", "accepted": True},
    {"id": "r2", "text": "really ", "segment_type": "remove", "explanation": "unnecessary word", "accepted": False},
    {"id": "r3", "text": "like cats.", "segment_type": "keep", "explanation": "", "accepted": True},
]

doc2 = DocumentState()
doc2.add_segments(remove_data)
assert doc2.get_working_text() == "I really like cats."
assert doc2.get_final_text() == "I like cats."

# accept the removal (delete the segment)
doc2.accept_segment("r2", "accept-remove")
assert doc2.get_working_text() == "I like cats."
assert len(doc2.segments) == 2

# undo — word comes back
doc2.undo()
assert doc2.get_working_text() == "I really like cats."

# reject the removal (keep the word)
doc2.accept_segment("r2", "keep-remove")
assert doc2.get_working_text() == "I really like cats."
assert doc2.segments[1].segment_type == "keep"

In [ ]:
# Test standalone add: "I like cats" where "really " is proposed for insertion
add_data = [
    {"id": "a1", "text": "I ", "segment_type": "keep", "explanation": "", "accepted": True},
    {"id": "a2", "text": "really ", "segment_type": "add", "explanation": "emphasis", "accepted": False},
    {"id": "a3", "text": "like cats.", "segment_type": "keep", "explanation": "", "accepted": True},
]

doc3 = DocumentState()
doc3.add_segments(add_data)
assert doc3.get_working_text() == "I like cats."
assert doc3.get_final_text() == "I like cats."

# accept the addition (convert to keep)
doc3.accept_segment("a2", "keep-add")
assert doc3.get_working_text() == "I really like cats."
assert doc3.segments[1].segment_type == "keep"

# undo — goes back to proposed add
doc3.undo()
assert doc3.get_working_text() == "I like cats."

# reject the addition (delete the segment)
doc3.accept_segment("a2", "reject-add")
assert doc3.get_working_text() == "I like cats."
assert len(doc3.segments) == 2

In [ ]:
# Test edge cases
doc4 = DocumentState()
doc4.add_segments(test_segments_data)

# invalid segment id
assert doc4.accept_segment("nonexistent", "add") == False
assert doc4.edit_segment("nonexistent", "hi") == False

# invalid action
assert doc4.accept_segment("s1", "banana") == False

# undo/redo at boundaries
assert doc4.undo() == False  # only one history entry, nothing to undo
assert doc4.redo() == False  # nothing to redo

# replace pair on last segment (no next segment)
assert doc4.accept_segment("s4", "add") == False

# replace pair on non-pair (two keeps)
assert doc4.accept_segment("s1", "remove") == False

# history truncation: make two edits, undo, make new edit, redo should fail
doc4.edit_segment("s1", "X ")
doc4.edit_segment("s1", "Y ")
doc4.undo()
assert doc4.segments[0].text == "X "
doc4.edit_segment("s1", "Z ")
assert doc4.redo() == False  # future was truncated

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()